In [1]:
import torch
from datasets import load_dataset, DatasetDict, Audio, ClassLabel

/home/pierre/Documents/Projects/PST4/AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SAMPLING_RATE = 16000
CHUNK_DURATION = 0.5
labels = ClassLabel(names=["other", "drone"])

In [ ]:
def load_and_prepare_hf_dataset(repo_name, split="test"):
    ds = load_dataset(repo_name, split=split)
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
    return ds

def load_and_prepare_audiofolder(path):
    ds = load_dataset("audiofolder", data_dir=path)["train"]
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
    return ds

def swap_labels(ds):
    def _swap(example):
        example["label"] = 1 - example["label"]
        return example
    return ds.map(_swap)

In [4]:
# -----------------------------
# Load datasets
# -----------------------------
ds_drone_test = load_and_prepare_hf_dataset("Usernameeeeee/drone_test")
ds_drone_test = ds_drone_test.remove_columns(["filename"])
ds_drone_test_2 = load_and_prepare_hf_dataset("Usernameeeeee/drone_test_2")
ds_drone_test_2 = ds_drone_test_2.remove_columns(["filename"])

ds_mic_parabole = load_and_prepare_audiofolder(
    "../../data/test/mic_parabole"
)
ds_parabole2 = load_and_prepare_audiofolder(
    "../../data/test/parabole2"
)
ds_parabole3 = load_and_prepare_audiofolder(
    "../../data/raw/test/28-01-2026/drone1"
)
ds_parabole4 = load_and_prepare_audiofolder(
    "../../data/raw/test/28-01-2026/drone2"
)
ds_parabole5 = load_and_prepare_audiofolder(
    "../../data/raw/test/28-01-2026/drone3"
)
ds_parabole6 = load_and_prepare_audiofolder(
    "../../data/raw/test/28-01-2026/drone4"
)
ds_parabole7 = load_and_prepare_audiofolder(
    "/home/pierre/Downloads/hibou_test/hibou_dataset"
)
print(ds_parabole7)

Dataset({
    features: ['audio', 'label'],
    num_rows: 7953
})


In [5]:
ds_drone_test = swap_labels(ds_drone_test)
ds_drone_test_2 = swap_labels(ds_drone_test_2)
ds_mic_parabole = swap_labels(ds_mic_parabole)
ds_parabole2 = swap_labels(ds_parabole2)
ds_parabole3 = swap_labels(ds_parabole3)
ds_parabole4 = swap_labels(ds_parabole4)
ds_parabole5 = swap_labels(ds_parabole5)
ds_parabole6 = swap_labels(ds_parabole6)
ds_parabole7 = swap_labels(ds_parabole7)

ds_drone_test = ds_drone_test.cast_column(
    "label",
    labels
)
ds_drone_test_2 = ds_drone_test_2.cast_column(
    "label",
    labels
)
ds_mic_parabole = ds_mic_parabole.cast_column(
    "label",
    labels
)
ds_parabole2 = ds_parabole2.cast_column(
    "label",
    labels
)
ds_parabole3 = ds_parabole3.cast_column(
    "label",
    labels
)
ds_parabole4 = ds_parabole4.cast_column(
    "label",
    labels
)
ds_parabole5 = ds_parabole5.cast_column(
    "label",
    labels
)
ds_parabole6 = ds_parabole6.cast_column(
    "label",
    labels
)
ds_parabole7 = ds_parabole7.cast_column(
    "label",
    labels
)

In [6]:
all_tests = DatasetDict({
    "drone_test": ds_drone_test,
    "drone_test_2": ds_drone_test_2,
    "parabole1": ds_mic_parabole,
    "parabole2": ds_parabole2,
    "parabole3": ds_parabole3,
    "parabole4": ds_parabole4,
    "parabole5": ds_parabole5,
    "parabole6": ds_parabole6,
    "parabole7": ds_parabole7,
})

In [12]:
def split_audio_into_chunks(audio_array, chunk_duration=CHUNK_DURATION, sampling_rate=SAMPLING_RATE):
    samples_per_chunk = int(chunk_duration * sampling_rate)
    num_chunks = audio_array.shape[-1] // samples_per_chunk
    chunks = [audio_array[i * samples_per_chunk:(i + 1) * samples_per_chunk]
              for i in range(num_chunks)]

    return chunks

def chunk_audio_batch(batch):
    all_audios = []
    all_sampling_rates = []
    all_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        audio_array = audio["array"]
        sampling_rate = audio["sampling_rate"]
        audio_array = torch.tensor(audio_array).float()
        chunks = split_audio_into_chunks(audio_array)

        all_audios.extend([chunk.numpy() for chunk in chunks])
        all_sampling_rates.extend([sampling_rate] * len(chunks))
        all_labels.extend([label] * len(chunks))

    return {
        "audio": all_audios,
        "label": all_labels,
    }


chunked_dataset = DatasetDict()
for split in all_tests.keys():
    print(f"Chunking split: {split}, original length: {len(all_tests[split])}")
    chunked_split = all_tests[split].map(
        chunk_audio_batch,
        batched=True,
        num_proc=20,
        batch_size=32,
        remove_columns=all_tests[split].column_names,
    )
    print(f"Generated split {split} length, {len(chunked_split)}")
    chunked_dataset[split] = chunked_split

Chunking split: drone_test, original length: 893
Generated split drone_test length, 893
Chunking split: drone_test_2, original length: 2805
Generated split drone_test_2 length, 2805
Chunking split: parabole1, original length: 123
Generated split parabole1 length, 122
Chunking split: parabole2, original length: 85
Generated split parabole2 length, 85
Chunking split: parabole3, original length: 275
Generated split parabole3 length, 275
Chunking split: parabole4, original length: 157
Generated split parabole4 length, 157
Chunking split: parabole5, original length: 83
Generated split parabole5 length, 83
Chunking split: parabole6, original length: 613
Generated split parabole6 length, 613
Chunking split: parabole7, original length: 7953
Generated split parabole7 length, 15906


In [13]:
chunked_dataset.push_to_hub("Hibou-Foundation/all_tests_ds_3_chunked")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  8.22ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 28.8MB / 28.8MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  2.26ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 62.2MB / 62.2MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 42.57ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 4.51MB / 4.51MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 48.17ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files 

CommitInfo(commit_url='https://huggingface.co/datasets/Hibou-Foundation/all_tests_ds_3_chunked/commit/71d4894bd127581ae3784d6951dccfafb8b06279', commit_message='Upload dataset', commit_description='', oid='71d4894bd127581ae3784d6951dccfafb8b06279', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Hibou-Foundation/all_tests_ds_3_chunked', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Hibou-Foundation/all_tests_ds_3_chunked'), pr_revision=None, pr_num=None)

In [14]:
from datasets import concatenate_datasets

concatenated_dataset = concatenate_datasets(
    [
        chunked_dataset["drone_test"],
        chunked_dataset["drone_test_2"],
        chunked_dataset["parabole1"],
        chunked_dataset["parabole2"],
        chunked_dataset["parabole3"],
        chunked_dataset["parabole4"],
        chunked_dataset["parabole5"],
        chunked_dataset["parabole6"],
        chunked_dataset["parabole7"],
    ],
)
len(concatenated_dataset)

20939

In [15]:
concatenated_dataset.push_to_hub("Hibou-Foundation/combined_test_dataset_v3_chunked")

Creating parquet from Arrow format: 100%|██████████| 4/4 [00:01<00:00,  3.05ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  67%|██████▋   |  204MB /  304MB,  356MB/s  
Processing Files (0 / 1):  77%|███████▋  |  233MB /  304MB,  286MB/s  
Processing Files (0 / 1):  78%|███████▊  |  236MB /  304MB,  218MB/s  
Processing Files (0 / 1):  80%|████████  |  245MB /  304MB,  183MB/s  
Processing Files (0 / 1):  83%|████████▎ |  252MB /  304MB,  159MB/s  
Processing Files (0 / 1):  86%|████████▌ |  261MB /  304MB,  143MB/s  
Processing Files (0 / 1):  88%|████████▊ |  268MB /  304MB,  129MB/s  
Processing Files (0 / 1):  90%|█████████ |  274MB /  304MB,  118MB/s  
Processing Files (0 / 1):  92%|█████████▏|  281MB /  304MB,  110MB/s  
Processing Files (0 / 1):  94%|█████████▍|  287MB /  304MB,  103MB/s  
Processing Files (0 / 1):  96%|█████████▋|  293MB /  304MB, 96.5MB/s  
Processing Files (0 / 1):  99%|█████████▉|  302MB /  304MB, 92.6MB/s  

CommitInfo(commit_url='https://huggingface.co/datasets/Hibou-Foundation/combined_test_dataset_v3_chunked/commit/b8e3ca29d6627986eeb7795e6229e1306f398437', commit_message='Upload dataset', commit_description='', oid='b8e3ca29d6627986eeb7795e6229e1306f398437', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Hibou-Foundation/combined_test_dataset_v3_chunked', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Hibou-Foundation/combined_test_dataset_v3_chunked'), pr_revision=None, pr_num=None)